# 第16章　少ないデータで戦う ― データ拡張・自己教師あり学習・合成データ

**『医療診断支援AI開発　実装編 ― 本格実装（実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-impl

## 拡張の「配合」を自動化する ― AutoAugmentからTrivialAugmentへ

In [ ]:
import random

# 医療向けに「危険な変換」を除いた候補集合で RandAugment を組む
SAFE_OPS = ["rotate", "translate_x", "translate_y",   # 幾何：軽い剛体変形
            "contrast", "brightness", "sharpness", "gaussian_blur"]  # 光学：撮影条件の模擬
# 除外例：posterize/solarize/invert（階調操作・反転はCT/X線の意味を壊す）、
#         強いshear/color（臓器配置・病変の見えを非現実にする）
def rand_augment(img, n=2, m=9):                       # m は 0..30 の強度
    for op in random.sample(SAFE_OPS, k=n):
        img = APPLY[op](img, magnitude=m)
    return img

## 手を動かす ― 擬似ラベルと一貫性を、一つの学習ループに組む

In [ ]:
import torch

tau, lam = 0.95, 1.0                       # 擬似ラベル採用の信頼度と、無ラベル項の重み
for x_l, y_l, x_u in loader:               # ラベル付き, 正解, ラベルなし
    loss_s = ce(model(weak(x_l)), y_l)     # 教師あり損失
    with torch.no_grad():
        p = model(weak(x_u)).softmax(1)    # 弱拡張で仮予測
        conf, pl = p.max(1)
        mask = conf >= tau                 # 自信のある無ラベルだけ使う
    logits_u = model(strong(x_u))          # 強拡張の予測を…
    loss_u = (ce_none(logits_u, pl) * mask).mean()   # …擬似ラベルへ合わせる
    optimizer.zero_grad(set_to_none=True)
    (loss_s + lam * loss_u).backward()
    optimizer.step()                       # ← これがないと、重みは一度も更新されない

## 拡散モデルで、病変を「描き加える」 ― 条件付き生成による拡張

In [ ]:
# マスク条件つき inpainting の骨格（潜在拡散を想定）
z = vae.encode(normal_image)                 # 画像を潜在空間へ
z_t = add_noise(z, timesteps[-1])             # 最終ステップまでノイズを入れた初期値
for t in reversed(timesteps):                 # 逆拡散で少しずつ生成
    eps = unet(z_t, t, cond=lesion_class,     # 病変クラスで条件づけ
               mask=lesion_mask)
    z_t = denoise_step(z_t, eps, t)
    z_t = z_t * lesion_mask + add_noise(z, t) * (1 - lesion_mask)  # 病変域だけ生成、外は原画像を維持
synth = vae.decode(z_t)

## 手を動かす ― 獲得関数を選び、次に何を塗るか決める

In [ ]:
import torch

@torch.no_grad()
def uncertainty_scores(model, pool_loader):
    ent, marg = [], []
    for x in pool_loader:
        p = model(x).softmax(1)
        t2 = p.topk(2, dim=1).values
        marg.append((t2[:,0] - t2[:,1]).cpu())               # 小さいほど不確実
        ent.append((-(p * p.clamp_min(1e-9).log()).sum(1)).cpu())  # 大きいほど不確実
    return torch.cat(ent), torch.cat(marg)

In [ ]:
import torch

# 不確実性と棄却の章の3関数を、章単位で動かせるように再掲
def enable_dropout(model):
    for m in model.modules():
        if m.__class__.__name__.startswith("Dropout"):
            m.train()

def snapshot_modes(model):
    return [(m, m.training) for m in model.modules()]

def restore_modes(snapshot):
    for m, flag in snapshot:
        m.train(flag)

@torch.no_grad()                                   # T回ぶんの計算グラフを持たない（3DではOOMの元）
def bald(model, x, T=20):
    saved = snapshot_modes(model)                  # 不確実性の章の enable_dropout / snapshot_modes / restore_modes
    try:
        model.eval(); enable_dropout(model)        # BatchNorm は固定し、Dropout だけ揺らす
        ps = torch.stack([model(x).softmax(1) for _ in range(T)])   # (T,B,C)
    finally:
        restore_modes(saved)                       # 呼ぶ前のモードへ必ず戻す
    mean = ps.mean(0)
    H_mean = -(mean * mean.clamp_min(1e-9).log()).sum(1)        # 平均予測の不確実さ
    mean_H = -(ps * ps.clamp_min(1e-9).log()).sum(2).mean(0)    # 各推論の不確実さの平均
    return H_mean - mean_H                                     # 大きいほど「割れている」